In [ ]:
import ipywidgets as widgets
from IPython.display import display
import requests
from google.colab import userdata

# NOTE: The original code had 'from Countrydetails import countries' which caused a ModuleNotFoundError.
# I've defined a list of major cities from Telangana for the dropdown.
global telangana_cities
telangana_cities = [
    "Hyderabad", "Warangal", "Nizamabad", "Khammam", "Karimnagar",
    "Ramagundam", "Mahbubnagar", "Siddipet", "Miryalaguda", "Suryapet",
    "Adilabad", "Nalgonda", "Jagtial", "Mancherial", "Kothagudem",
    "Bodhan", "Sangareddy", "Vikarabad", "Medak", "Wanaparthy",
    "Peddapalli", "Secunderabad"
]

# Widgets for the UI
city_selector = widgets.Combobox(
    options=telangana_cities,
    value='',
    placeholder='Select or type city name',
    description='City (Telangana):',
    ensure_option=False, # Allows typing values not in options
    disabled=False
)

get_weather_button = widgets.Button(
    description='Get Weather',
    disabled=False,
    button_style='info', # 'success', 'info', 'warning', 'danger' or ''
    tooltip='Click to get weather',
    icon='cloud'
)

result_output = widgets.Output()

def get_weather_ipywidgets(b):
    with result_output:
        result_output.clear_output()
        city = city_selector.value # Get value from the Combobox

        if not city:
            print("Please enter a city name.")
            return

        # Retrieve API key securely from Colab's userdata secrets
        api_key = userdata.get('OPENWEATHER_API_KEY') # You need to set this in Colab Secrets
        if not api_key:
            print("Error: OpenWeatherMap API key not found in Colab secrets. Please set 'OPENWEATHER_API_KEY'.")
            return

        url = f'http://api.openweathermap.org/data/2.5/weather?q={city}&APPID={api_key}'
        try:
            response = requests.get(url)
            response.raise_for_status()  # Raise an exception for HTTP errors
            data = response.json()

            temperature_kelvin = data['main']['temp']
            temperature_celsius = temperature_kelvin - 273.15
            temperature_fahrenheit = (temperature_kelvin - 273.15) * 9/5 + 32
            humidity = data['main']['humidity']
            weather_condition = data['weather'][0]['description']
            wind_speed = data['wind']['speed']
            pressure = data['main']['pressure']
            country_code = data['sys']['country']
            city_with_country = f"{city}, {country_code}"

            print(f"City: {city_with_country}")
            print(f"Temperature: {temperature_celsius:.2f} °C / {temperature_fahrenheit:.2f} °F")
            print(f"Humidity: {humidity}%")
            print(f"Weather Condition: {weather_condition}")
            print(f"Wind Speed: {wind_speed} m/s")
            print(f"Pressure: {pressure} hPa")

        except requests.exceptions.RequestException as e:
            print(f"Error fetching data: {e}")
        except KeyError as e:
            print(f"Error: Data format incorrect or city not found. Please try a different city. Details: {e}")

# Link the button to the function
get_weather_button.on_click(get_weather_ipywidgets)

# Display the widgets
display(widgets.VBox([
    widgets.Label("Weather App (Telangana Cities)", style={'font_weight': 'bold', 'font_size': '24px'}),
    city_selector, # Use the new Combobox
    get_weather_button,
    result_output
]))
